In [1]:
# Use Jupyter magic to install packages
%pip install pandas seaborn scikit-learn statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd 

df = pd.read_stata("ed2022-stata.dta", convert_categoricals = False)
df[['WAITTIME', 'RACERETH', 'IMMEDR', 'AGE', 'SEX', 'ARRTIME', 'ARREMS', 'PAYTYPER']].describe()

,WAITTIME,RACERETH,IMMEDR,AGE,SEX,ARREMS,PAYTYPER
count,16025.000000,16025.000000,16025.000000,16025.000000,16025.000000,16025.000000,16025.000000
mean,28.363557,1.704961,0.384462,40.266396,1.471763,1.506771,1.336599
std,61.939166,0.891837,5.179554,24.968400,0.499218,1.778335,3.518800
min,-9.000000,1.000000,-9.000000,0.000000,1.000000,-9.000000,-9.000000
25%,1.000000,1.000000,-8.000000,20.000000,1.000000,2.000000,1.000000
50%,10.000000,1.000000,3.000000,38.000000,1.000000,2.000000,2.000000
75%,31.000000,2.000000,4.000000,60.000000,2.000000,2.000000,3.000000
max,1280.000000,4.000000,7.000000,94.000000,2.000000,2.000000,7.000000


In [3]:
#Check ARRTIME
print(df["ARRTIME"].dtype)
df["ARRTIME"].value_counts(dropna=False).head(20)

str


ARRTIME
-9      243
1124     29
1440     29
1309     27
1116     27
1224     27
1824     26
1052     25
1202     25
1222     25
1312     24
1835     24
1000     24
1630     24
1530     24
1110     24
1700     24
1212     24
1010     24
2100     24
Name: count, dtype: int64

In [4]:
#Convert ARRTIME to numeric, coercing errors to NaN
df["ARRTIME"] = pd.to_numeric(df["ARRTIME"], errors="coerce")
df["ARRTIME"].isna().sum()

np.int64(0)

In [5]:
df['ARRTIME'] = df['ARRTIME'].replace([-9], pd.NA)
print(df['ARRTIME'].isna().sum())
df['arrival_hour'] = (df['ARRTIME'] // 100).astype('Int64')

243


In [6]:
#Check for missing values in ARREMS
df["ARREMS"] = df["ARREMS"].replace([-9], pd.NA)
df["ARREMS"].isna().sum()

np.int64(106)

In [7]:
#Check for missing values in WAITTIME
df["WAITTIME"] = df["WAITTIME"].replace([-9,], pd.NA)
df["WAITTIME"].isna().sum()

np.int64(2173)

In [8]:
#Check for patients not seen by a physician in WAITTIME
df["NOT_SEEN_WAITTIME"] = df["WAITTIME"] == -7
df["NOT_SEEN_WAITTIME"].sum()

/var/folders/zz/bs7v7lqn5zvbn2sd5r05znrw0000gn/T/ipykernel_18070/3682692509.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["NOT_SEEN_WAITTIME"] = df["WAITTIME"] == -7


np.int64(580)

In [9]:
#% of missing values in ETHNICITY (unimputed)
df['ETHUN'].value_counts(dropna=False)
df['eth_missing'] = df['ETHUN'] == -9
df['eth_missing'].sum()
df['eth_missing'].mean() * 100

/var/folders/zz/bs7v7lqn5zvbn2sd5r05znrw0000gn/T/ipykernel_18070/1977338001.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['eth_missing'] = df['ETHUN'] == -9


np.float64(14.558502340093604)

In [10]:
#% of missing values in RACE (unimputed)
df['RACEUN'].value_counts(dropna=False)
df['race_missing'] = df['RACEUN'] == -9
df['race_missing'].sum()
df['race_missing'].mean() * 100

/var/folders/zz/bs7v7lqn5zvbn2sd5r05znrw0000gn/T/ipykernel_18070/2485565852.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['race_missing'] = df['RACEUN'] == -9


np.float64(18.858034321372855)

In [11]:
# Check if missingness rates in WAIT are similar across race groups
df.groupby('RACERETH')['WAITTIME'].apply(lambda x: x.isna().mean())

RACERETH
1    0.128829
2    0.130611
3    0.154079
4    0.179300
Name: WAITTIME, dtype: float64

In [12]:
#Check for missing values in IMMEDIANCY WITH WHICH PATIENTS SHOULD BE SEEN
df["IMMEDR"] = df["IMMEDR"].replace([-9, -8, 0, 7], pd.NA)
df["IMMEDR"].isna().sum()

np.int64(5818)

In [13]:
#Check PAYTYPER for missing values
df['PAYTYPER'] = df['PAYTYPER'].replace([-9, -8], pd.NA)
df['PAYTYPER'].isna().sum()

np.int64(1737)

In [14]:
# clean WAITTIME variable by dropping rows with missing values
df = df.copy()
df_model = df.dropna(subset=['WAITTIME', 'IMMEDR', 'ARREMS', 'ARRTIME', 'PAYTYPER'])
print(df_model.shape)

(8247, 917)


In [15]:
model_cols = ['WAITTIME', 'RACERETH', 'IMMEDR', 'AGE', 'SEX', 'ARREMS',
              'PAYTYPER', 'arrival_hour', 'NOT_SEEN_WAITTIME']
df_model = df_model[model_cols]
print(df_model.shape)

(8247, 9)


In [16]:
df_model.to_pickle('nhamcs_clean.pkl')